# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường

Mã thật nằm trong `lab/` (`labels.py`, `vlm.py`, `reward.py`, `viz.py`); notebook này gọi các hàm đó theo từng bước của `GUIDE.md`.
Mọi kết quả chạy (adapter, dự đoán, bằng chứng) được lưu lên **Google Drive** để không mất khi Colab ngắt. Chạy lại một ô đã xong sẽ **bỏ qua** phần đã làm.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/phuongling2428-wq/K4-Track-4-Day10-VisionLanguageModel.git'
import os
if not os.path.exists('/content/lab10'):
    !git clone -q {REPO_URL} /content/lab10
else:
    !git -C /content/lab10 pull -q     # lấy bản sửa mới nhất
%cd /content/lab10
!pip -q install -r requirements-colab.txt
# Colab cài sẵn torchao 0.10; peft 0.21 báo ImportError khi gắn LoRA lên nn.Linear thường (ViT ở E3). Lab không dùng torchao.
!pip -q uninstall -y torchao

In [ ]:
import json, shutil, subprocess, functools
from pathlib import Path
import torch

DRIVE = Path('/content/drive/MyDrive/day10_v2')          # nơi lưu adapter, dự đoán, bằng chứng
RUNS, PREDS = DRIVE / 'runs', DRIVE / 'preds'
EVID, FIGS = Path('report/evidence'), Path('report/figures')
for d in (RUNS, PREDS, EVID, FIGS, DRIVE / 'report'):
    d.mkdir(parents=True, exist_ok=True)

def sync_report():
    """Chép report/ lên Drive sau mỗi lần chấm (clone trên Colab sẽ mất khi ngắt)."""
    shutil.copytree('report', DRIVE / 'report', dirs_exist_ok=True)

# Khôi phục bằng chứng đã có từ lần chạy trước (nếu session bị ngắt)
if (DRIVE / 'report').exists():
    shutil.copytree(DRIVE / 'report', 'report', dirs_exist_ok=True)
print(torch.__version__, torch.cuda.get_device_name(0))

## 1. Dữ liệu: tải và chia split

In [ ]:
# Tải zip công khai của dataset qua URL trực tiếp (không cần tài khoản/khoá Kaggle).
# Nếu URL lỗi: tải zip về tay, đưa lên Drive rồi dùng --zip /content/drive/MyDrive/<file>.zip
DATA_URL = 'https://www.kaggle.com/api/v1/datasets/download/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes'
if not Path('data/splits/meta.json').exists():
    !python scripts/prepare_data.py --zip "{DATA_URL}"
!python -m pytest -q

In [ ]:
from scripts.labcommon import read_jsonl
from lab import viz

rows, leak = viz.split_stats()
assert not leak, 'có phiên nằm ở hai split!'
META = json.loads(Path('data/splits/meta.json').read_text())
print('dataset_version =', META['dataset_version'])
train_e, val_e, test_e = (read_jsonl(f'data/splits/{s}.jsonl') for s in ('train', 'val', 'test'))
print(train_e[0])

In [ ]:
# 6 ảnh có box: trắng = box YOLO gốc, màu = box sau khi đi qua JSON 0–1000 rồi quy ngược về pixel
import random
sample = random.Random(0).sample([e for e in train_e if len(e['objects']) >= 2], 6)
g = viz.grid([viz.draw_gt(e) for e in sample], cols=3, captions=[e['image_id'][:40] for e in sample])
g.save(FIGS / 'gt_samples.jpg', quality=88); g

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from lab.labels import PROMPT, build_prompt, objects_to_answer, yolo_to_box, box_to_pixel
e = sample[0]
print(PROMPT, '\n')
print('objects:', e['objects'])
print('answer :', objects_to_answer(e['objects']))
# Kiểm tra khứ hồi: 0–1000 → pixel phải trùng box YOLO gốc (sai số làm tròn < 1 px)
for o in e['objects']:
    xc, yc, w, h = o['yolo']
    ref = ((xc - w/2) * 640, (yc - h/2) * 360, (xc + w/2) * 640, (yc + h/2) * 360)
    back = box_to_pixel(yolo_to_box(o['label'], o['yolo'])['bbox_2d'])
    assert max(abs(a - b) for a, b in zip(ref, back)) < 1.0, (ref, back)
print('khứ hồi OK')

## 3. E0: zero-shot

In [ ]:
from lab import vlm
model, processor = vlm.load_model()

# Câu 1: số tham số ViT/projector (fp16, phải ≈ 415M / 27,3M / 81,8M; nếu chỉ bằng một nửa là ViT đã bị lượng tử)
def n_params(f):
    return sum(p.numel() for n, p in model.named_parameters() if f(n))
print('ViT (gồm projector):', f"{n_params(lambda n: 'visual' in n):,}")
print('  visual.merger     :', f"{n_params(lambda n: 'visual.merger' in n):,}")
print('  deepstack_mergers :', f"{n_params(lambda n: 'deepstack_merger_list' in n):,}")
print('dtype ViT:', next(model.model.visual.parameters()).dtype,
      '· kiểu lớp q_proj LLM:', type(model.model.language_model.layers[0].self_attn.q_proj).__name__)
print(f'VRAM sau khi nạp: {torch.cuda.memory_allocated()/2**30:.2f} GB')

In [ ]:
def score(pred, name, out, coord='norm1000', split='test', meta=None, splits_dir='data/splits'):
    cmd = ['python', 'scripts/score.py', '--pred', str(pred), '--name', name, '--out', str(out),
           '--coord', coord, '--split', split, '--splits-dir', str(splits_dir)]
    for k, v in (meta or {}).items():
        cmd += ['--meta', f'{k}={v}']
    print(subprocess.run(cmd, capture_output=True, text=True, check=True).stdout)
    sync_report()
    return json.loads(Path(out).read_text())

# Gỡ lỗi nhanh trên 40 ảnh val trước khi chạy cả test
# score.py chấm cả split, nên dựng một thư mục split chỉ chứa đúng 40 ảnh này
from scripts.labcommon import write_jsonl
VAL40 = Path('/content/val40'); write_jsonl(VAL40 / 'val.jsonl', val_e[:40])
shutil.copy('data/splits/meta.json', VAL40 / 'meta.json')
vlm.predict(model, processor, val_e[:40], 'data', PROMPT, PREDS / 'e0_val40.jsonl')
score(PREDS / 'e0_val40.jsonl', 'E0-val40', DRIVE / 'debug_e0_val40.json', split='val', splits_dir=VAL40);

In [ ]:
vlm.predict(model, processor, test_e, 'data', PROMPT, PREDS / 'e0.jsonl')
r0 = score(PREDS / 'e0.jsonl', 'E0', EVID / 'e0.json')
for r in read_jsonl(PREDS / 'e0.jsonl')[:5]:
    print(repr(r['raw'][:300]), '\n')

In [ ]:
# Câu 2: phân loại lỗi của E0
from scripts.score import parse_output
from collections import Counter
cnt = Counter()
for r in read_jsonl(PREDS / 'e0.jsonl'):
    raw, p = r['raw'], parse_output(r['raw'])
    if p is None: cnt['JSON hỏng'] += 1; continue
    if '```' in raw: cnt['bọc ```json'] += 1
    if any(b['label'] not in ('pothole', 'crack', 'manhole') for b in p): cnt['nhãn ngoài 3 lớp'] += 1
    if any(max(b['bbox']) > 1000 for b in p): cnt['toạ độ > 1000 (thang pixel?)'] += 1
    if not p: cnt['trả []'] += 1
print(cnt)
del model; vlm.cleanup()

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(vlm.MODEL_ID); processor.tokenizer.padding_side = 'left'
ds = vlm.GroundingDataset(train_e, 'data', processor, PROMPT)
ex = ds[0]
pad = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
print('số token ảnh:', int((ex['input_ids'] == pad).sum()), '· grid', ex['image_grid_thw'].tolist())
print('tổng token:', ex['input_ids'].shape[1], '· token có loss:', int((ex['labels'] != -100).sum()))
print('phần được học:', repr(processor.tokenizer.decode(ex['input_ids'][ex['labels'] != -100])))
print('đáp án       :', repr(ds.answer(train_e[0])))
batch = vlm.collate([ds[0], ds[1]], processor.tokenizer.pad_token_id)
print({k: tuple(v.shape) for k, v in batch.items()})

## 5. Setup LoRA / đóng băng cho E1, E2, E3

`vlm.apply_setup(model, targets, train_projector)`: đóng băng mọi tham số, bật gradient checkpointing, gắn LoRA r=16 theo regex (`LLM_TARGETS` 252 module; thêm `VIT_TARGETS` 96 module ở E3), rồi với E2/E3 mở `visual.merger` + `visual.deepstack_merger_list` ở float32. Số module khớp và số tham số được ghi vào `runs/eN/setup.json`.

## 6. Vòng huấn luyện

`vlm.train`: AdamW lr 2e-4, warmup 5% + cosine, fp16 autocast + GradScaler, batch 2 × tích luỹ 4, 300 bước, clip grad 1.0, dừng nếu loss không hữu hạn; lưu adapter **và** `extra_trainable.pt` (projector) mỗi 100 bước; ghi `run.json`.

In [ ]:
def run_experiment(key, targets='llm', train_projector=False, r=16, alpha=32, train_fraction=1.0,
                   coord='norm1000', max_steps=300, evidence_name=None):
    """Huấn luyện (nếu chưa có run.json trên Drive) → nạp lại → suy luận test → chấm."""
    if (EVID / f'{key}.json').exists():          # đã xong ở phiên trước: bỏ qua
        print(key, 'đã có bằng chứng, bỏ qua'); return json.loads((EVID / f'{key}.json').read_text())
    out = RUNS / key
    prompt = build_prompt(coord)
    if not (out / 'run.json').exists():
        model, processor = vlm.load_model()
        model, info = vlm.apply_setup(model, targets, train_projector, r=r, alpha=alpha)
        (out).mkdir(parents=True, exist_ok=True)
        (out / 'setup.json').write_text(json.dumps(info, indent=2))
        entries = vlm.subsample(train_e, train_fraction)
        ds = vlm.GroundingDataset(entries, 'data', processor, prompt, coord)
        run = vlm.train(model, ds, functools.partial(vlm.collate, pad_id=processor.tokenizer.pad_token_id),
                        out, max_steps=max_steps)
        print(run)
        del model; vlm.cleanup()
    run = json.loads((out / 'run.json').read_text())
    pred = PREDS / f'{key}.jsonl'
    model, processor = vlm.load_trained(out)
    vlm.predict(model, processor, test_e, 'data', prompt, pred)
    del model; vlm.cleanup()
    meta = {k: run[k] for k in ('trainable_params', 'peak_vram_gb', 'train_minutes')}
    return score(pred, evidence_name or key.upper(), EVID / f'{key}.json', coord=coord, meta=meta)

def plot_loss(*keys):
    import matplotlib.pyplot as plt
    for k in keys:
        log = json.loads((RUNS / k / 'train_log.json').read_text())
        plt.plot([x['step'] for x in log], [x['loss'] for x in log], label=k.upper())
    plt.xlabel('bước'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=.3)
    plt.savefig(FIGS / 'loss.png', dpi=120, bbox_inches='tight'); plt.show()

## 7. Suy luận và chấm E1–E3

In [ ]:
r1 = run_experiment('e1', targets='llm', train_projector=False)

In [ ]:
r2 = run_experiment('e2', targets='llm', train_projector=True)

In [ ]:
r3 = run_experiment('e3', targets='llm+vit', train_projector=True)
plot_loss('e1', 'e2', 'e3')
for k in ('e1', 'e2', 'e3'):
    print(k, json.loads((RUNS / k / 'setup.json').read_text()))

In [ ]:
# Câu 3: chấm lại dự đoán E1 (thang 0–1000) như thể là pixel → F1 sụp. Không đặt vào evidence/.
Path('report/extra').mkdir(exist_ok=True)
score(PREDS / 'e1.jsonl', 'E1-as-pixel', 'report/extra/e1_scored_as_pixel.json', coord='pixel')

## 8. E4: ablation (đổi đúng một biến so với setup tốt nhất)

Mặc định chọn `coord_scale` (nối thẳng với câu 3). Đổi `E4_VAR` thành `'rank'` hoặc `'data_size'` nếu muốn.
Giá trị gốc của setup tốt nhất được chấm lại từ chính dự đoán của nó (không cần chạy lại).

In [ ]:
SETUPS = {'e1': dict(targets='llm', train_projector=False),
          'e2': dict(targets='llm', train_projector=True),
          'e3': dict(targets='llm+vit', train_projector=True)}
ev = {k: json.loads((EVID / f'{k}.json').read_text()) for k in SETUPS}
BEST = max(ev, key=lambda k: ev[k]['micro']['f1'])
print('setup tốt nhất:', BEST, {k: round(v['micro']['f1'], 3) for k, v in ev.items()})

E4_VAR = 'coord_scale'            # 'rank' | 'data_size' | 'coord_scale'
base = SETUPS[BEST]
if E4_VAR == 'coord_scale':
    base_val, new_val, new_kw = 'norm1000', 'pixel', dict(coord='pixel')
elif E4_VAR == 'rank':
    base_val, new_val, new_kw = 16, 64, dict(r=64, alpha=128)   # (hoặc 4 → r=4, alpha=8)
else:
    base_val, new_val, new_kw = 1.0, 0.1, dict(train_fraction=0.1)

base_meta = {k: ev[BEST]['meta'][k] for k in ('trainable_params', 'peak_vram_gb', 'train_minutes')}
score(PREDS / f'{BEST}.jsonl', f'E4 {E4_VAR}={base_val}', EVID / f'e4_{E4_VAR}_{base_val}.json', meta=base_meta)
r4 = run_experiment(f'e4_{E4_VAR}_{new_val}', **base, **new_kw, evidence_name=f'E4 {E4_VAR}={new_val}')

## 9. E5 (tuỳ chọn, +10): GRPO với reward IoU

In [ ]:
!python -m pytest -q lab_tests/test_reward.py

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from lab.reward import grpo_reward, gt_boxes_norm1000
from PIL import Image

E5_DIR = RUNS / 'e5'
grpo_entries = vlm.subsample(train_e, 0.2, seed=5)
def gen():
    for e in grpo_entries:
        yield {'prompt': [{'role': 'user', 'content': PROMPT}],
               'image': Image.open(Path('data') / e['image_path']).convert('RGB'),
               'gt': json.dumps([[lab, list(bb)] for lab, bb in gt_boxes_norm1000(e['objects'])])}
grpo_ds = Dataset.from_generator(gen)

if not (E5_DIR / 'run.json').exists():
    model, processor = vlm.load_trained(RUNS / BEST, is_trainable=True)   # khởi tạo từ adapter tốt nhất
    model.enable_input_require_grads()
    cfg = GRPOConfig(output_dir=str(E5_DIR), num_generations=4, max_completion_length=128, max_steps=60,
                     learning_rate=5e-6, beta=0.0, per_device_train_batch_size=4, gradient_accumulation_steps=2,
                     fp16=True, bf16=False, gradient_checkpointing=True, use_vllm=False, logging_steps=1,
                     save_strategy='no', report_to='none', remove_unused_columns=False, log_completions=False)
    trainer = GRPOTrainer(model=model, reward_funcs=grpo_reward, args=cfg, train_dataset=grpo_ds,
                          processing_class=processor)
    torch.cuda.reset_peak_memory_stats(); import time; t0 = time.time()
    trainer.train()
    model.save_pretrained(E5_DIR / 'adapter')
    if (RUNS / BEST / vlm.EXTRA_FILE).exists():       # projector không đổi trong GRPO: dùng lại của BEST
        shutil.copy(RUNS / BEST / vlm.EXTRA_FILE, E5_DIR / vlm.EXTRA_FILE)
    run = {'trainable_params': sum(p.numel() for p in model.parameters() if p.requires_grad),
           'peak_vram_gb': round(torch.cuda.max_memory_allocated() / 2**30, 2),
           'train_minutes': round((time.time() - t0) / 60, 1), 'init': BEST}
    (E5_DIR / 'run.json').write_text(json.dumps(run, indent=2))
    Path('report/e5_grpo_log.json').write_text(json.dumps(trainer.state.log_history, indent=1))
    del trainer, model; vlm.cleanup()

In [ ]:
run = json.loads((E5_DIR / 'run.json').read_text())
if not Path('report/extra/e5.json').exists():
    model, processor = vlm.load_trained(E5_DIR)
    vlm.predict(model, processor, test_e, 'data', PROMPT, PREDS / 'e5.jsonl')
    del model; vlm.cleanup()
r5 = score(PREDS / 'e5.jsonl', 'E5', 'report/extra/e5.json',
           meta={k: run[k] for k in ('trainable_params', 'peak_vram_gb', 'train_minutes')})

# Dấu hiệu reward hacking: tỉ lệ trả [] và box khổng lồ (>50% khung hình), so với SFT tốt nhất
def hack_stats(path):
    rows = [parse_output(r['raw']) for r in read_jsonl(path)]
    ok = [p for p in rows if p is not None]
    boxes = [b for p in ok for b in p]
    giant = sum((b['bbox'][2]-b['bbox'][0]) * (b['bbox'][3]-b['bbox'][1]) > 0.5e6 for b in boxes)
    return {'empty_rate': round(sum(not p for p in ok) / len(rows), 3), 'boxes/img': round(len(boxes) / len(rows), 2),
            'giant_boxes': giant}
print('SFT', BEST, hack_stats(PREDS / f'{BEST}.jsonl'))
print('E5     ', hack_stats(PREDS / 'e5.jsonl'))
log = json.loads(Path('report/e5_grpo_log.json').read_text())
print([round(x['reward'], 3) for x in log if 'reward' in x])
sync_report()

## 10. Ảnh minh hoạ và bảng tổng hợp

In [ ]:
for key in ('e0', BEST):
    for r in viz.make_figures(PREDS / f'{key}.jsonl', FIGS, key):
        print(r['file'], r['kind'], f"TP={r['tp']} FP={r['fp']} FN={r['fn']}", r['gt'], '→', r['pred'])
from IPython.display import Image as IPImage, display
display(IPImage(str(FIGS / f'{BEST}_grid.jpg')))
sync_report()

In [ ]:
!python -m lab.report_table
!python scripts/check_submission.py

In [ ]:
# Tải report/ về máy rồi chép vào repo local (hoặc lấy từ Drive: MyDrive/day10/report)
!zip -qr /content/report.zip report
from google.colab import files
files.download('/content/report.zip')